<div style="border:2px solid #B3241C; background:#FDECEA; border-radius:8px; padding:16px 20px;">
  <div style="color:#B3241C; font-size:26px; font-weight:800; line-height:1.25; margin-bottom:8px;">
    STOP &mdash; do this first
  </div>
  <div style="font-size:19px; color:#1F2328; line-height:1.5;">
    <b>File &rarr; Save a copy in Drive</b>, then work in <b>your</b> copy.<br>
    If you skip this step, nothing you type here will be saved.
  </div>
</div>

# K513 Final Project · Notebook 2 — Understanding the model-ready table

**Your goal in this notebook:** describe the model-ready table, one column at a time and then in
pairs, and find out which columns are related to **late delivery** — the target Notebook 3 predicts.

**How this notebook works**

- It runs on the provided model-ready table (one row per delivered order), so you can start before
  Notebook 1 is finished. It is the same table Notebook 1 builds.
- Cells with `____` are yours to complete. Everything else runs as given.
- **📊 Slide N** tells you where each answer goes in the deck template.
- Use the exact statistical terms: mean, median, standard deviation, interquartile range (IQR),
  skew, outlier, correlation.

## 0 · Load the table

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.precision', 3)
pd.set_option('display.max_columns', 40)

BASE = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/FP"
DATES = ['order_purchase_timestamp', 'order_estimated_delivery_date',
         'order_delivered_carrier_date', 'order_delivered_customer_date']

df = pd.read_csv(f"{BASE}/olist_model_ready.csv.gz", parse_dates=DATES)
print(df.shape)

### The columns

| Column | What it is |
|---|---|
| `order_id`, `customer_unique_id`, `seller_id` | identifiers — the order, the customer, the seller of the first item line |
| `order_purchase_timestamp` | when the order was placed |
| `order_estimated_delivery_date` | the delivery date promised to the customer at checkout |
| `order_delivered_carrier_date` | when the seller handed the parcel to the carrier |
| `order_delivered_customer_date` | when the parcel reached the customer |
| `customer_city`, `customer_state` | where the customer is |
| `seller_city`, `seller_state` | where the seller of the first item line is |
| `same_city`, `same_state` | 1 if the customer and that seller are in the same city / state |
| `category` | product category (English) of the first item line; `unknown` if missing |
| `n_items` | number of item lines in the order |
| `n_diff_products`, `n_diff_sellers` | number of different products / sellers in the order |
| `revenue` | sum of item prices (BRL) |
| `freight` | sum of shipping charges (BRL) |
| `total_weight_g` | total weight of all items, grams |
| `max_length_cm`, `max_height_cm`, `max_width_cm` | the largest length / height / width of any item in the order |
| `payment_type` | payment method of the largest payment |
| `payment_installments` | the largest number of installments |
| `payment_value` | total paid (BRL) — item prices plus freight, plus any installment interest |
| `n_reviews` | number of reviews on the order |
| `review_score` | score of the latest review, 1–5 |
| `n_days_promised` | days from purchase to the promised delivery date |
| `n_days_to_carrier` | days from purchase to handing the parcel to the carrier |
| `n_days_ahead` | days the order arrived **before** the promised date; negative = late |
| `late` | **target** — 1 if the order arrived after the promised date, else 0 |

---
## 1 · The table

**Q1.1** How many rows and columns? What is one row? What is the **target**, and what share of the
orders were late (the **base rate**)?

In [ ]:
print("rows, columns:", df.shape)
print("late orders:  ", df['____'].sum())
print(f"base rate:     {df['late'].____():.3f}")
df.head(3)

**Q1.2** Which columns have missing values, and how many? Look at the rows involved. Then count
the orders whose `total_weight_g` is **0** — why would an order weigh nothing?

In [ ]:
print("Columns with missing values and number of missing values:")
missing = df.isna().sum()
print(missing[missing > 0], "\n")
print("orders with total_weight_g == 0:", (df['total_weight_g'] == ____).sum())
print("\norders without length or weight information:")
df[df['total_weight_g'] == 0][['order_id', 'n_items', 'category', 'total_weight_g',
                               'max_length_cm', 'max_height_cm', 'max_width_cm']]

*Hint for the zero weights: Notebook 1 used `aggfunc='sum'` for weight. What does `sum` return when
every value it adds is missing? Do all five have the same explanation? One has a category and dimensions.*

> 📊 **Slide 14** — rows, columns, what one row is, the target and its base rate, and the missing
> values with what they mean.

---
## 2 · Column types

**Q2.1** Sort the columns into four lists. Identifiers and dates are done for you. A column holding 0
or 1 that marks a group (`same_state`, `late`) is **categorical**, even though it is stored as a
number.

In [ ]:
IDENTIFIERS = ['order_id', ____]
DATE_COLUMNS = DATES

CONTINUOUS = ['n_items', ____]

CATEGORICAL = ['customer_state', ____]

assert len(IDENTIFIERS + DATE_COLUMNS + CONTINUOUS + CATEGORICAL) == df.shape[1]
print(len(CONTINUOUS), "continuous,", len(CATEGORICAL), "categorical")

> 📊 **Slide 15** — the four lists, and one sentence on any column you found hard to classify.

---
## 3 · One continuous column at a time

For each column: a **histogram** and a **boxplot**, the summary statistics, and the **skew**. Then
describe it in four parts — **center** (mean vs median), **spread** (standard deviation, IQR),
**shape** (symmetric or skewed, which way), and **outliers**. The first one is done for you.

**Q3.1** `revenue`

In [ ]:
def describe_column(col):
    """Histogram and boxplot side by side, then the numbers."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    sns.histplot(data=____, x=____, bins=60, ax=axes[0])
    sns.boxplot(data=____, x=____, ax=axes[1])
    plt.show()
    stats = df[col].describe()
    stats['IQR'] = stats['75%'] - stats['25%']
    stats['skew'] = df[col].skew()
    print(stats.round(2))

describe_column('revenue')

**Q3.2** `freight`

In [ ]:
describe_column('____')

> 📊 **Slide 16** — `revenue` and `freight`: one chart each, and center, spread, shape and outliers
> for each.

**Q3.3** `n_days_promised` and `total_weight_g`

In [ ]:
describe_column('____')

In [ ]:
describe_column('____')

> 📊 **Slide 17** — the same four descriptions for `n_days_promised` and `total_weight_g`.

**Q3.4** Now compare every column in your `CONTINUOUS` list at once. The table sorts them from the
largest skew (most right-skewed) to the most negative skew (most left-skewed).

In [ ]:
overview = pd.DataFrame({
    'mean':   df[CONTINUOUS].mean(),
    'median': df[CONTINUOUS].median(),
    'skew':   df[CONTINUOUS].____(),
}).sort_values('skew', ascending=False)
overview.round(2)

**(a)** Look at the five variables at the top of the table. Some have a long tail of large values: a few
orders cost far more than the rest. Others are counts where almost every order has the same value and
only a few orders differ. Which of the five are which? Run `value_counts()` on a count column to check.

In [ ]:
df['____'].value_counts()

**(b)** Which columns have a negative skew? For `n_days_ahead`, which orders are in the long left tail?
Run `describe_column` on it to see the tail.

**(c)** Which column has the skew closest to 0? Is its mean close to its median? Would you call it
symmetric?

In [ ]:
describe_column('____')

> 📊 **Slide 18** — the skew table, and one sentence each for (a), (b) and (c).

---
## 4 · Categorical columns

**Q4.1** For `late`, `payment_type`, `customer_state` (top 10) and `category` (top 10): the counts,
the percentages, and a bar chart.

In [ ]:
def describe_category(col, top=10):
    counts = df[col].value_counts()
    table = pd.DataFrame({'orders': counts, 'share': (counts / len(df)).round(3)}).head(top)
    print(table, "\n")
    plt.figure(figsize=(6, 0.35 * len(table) + 1))
    sns.countplot(data=____, y=____, order=table.index)
    plt.show()

describe_category('late')

In [ ]:
describe_category('____')

In [ ]:
describe_category('____')

In [ ]:
describe_category('____')

> 📊 **Slide 19** — the four charts (or the two most informative) with their key percentages.

---
## 5 · Pairs of continuous columns

**Q5.1** Make a scatterplot for each pair, and compute the **correlation**.
Describe each relationship: direction, strength, and whether it is a straight line.

- `revenue` and `freight`
- `total_weight_g` and `freight`

In [ ]:
def describe_pair(x, y):
    sns.scatterplot(data=____, x=____, y=____, alpha=0.2, s=10)
    plt.show()
    print(f"correlation {df[x].corr(df[y]):+.3f}")

describe_pair('revenue', 'freight')

In [ ]:
describe_pair('____', 'freight')

**Q5.2** The heatmap shows the correlation of every pair of continuous columns at once.
Which pairs are strongly correlated, and why would you expect that from what the columns measure?

In [ ]:
plt.figure(figsize=(11, 9))
sns.heatmap(df[CONTINUOUS].corr(), annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1)
plt.show()

> 📊 **Slide 20** — the two scatterplots with their correlations, and the strongest pairs in the
> heatmap with a reason for each.

---
## 6 · Late delivery and reviews

**Q6.1** Is late delivery related to bad reviews? Compare the mean review score, and the share of
bad reviews (score 1 or 2), for late and on-time orders.

In [ ]:
df['bad_review'] = (df['review_score'] <= ____).astype(int)
df.groupby('late')[['review_score', 'bad_review']].mean().round(3)

**Q6.2** Compute the correlation of `n_days_ahead` with `review_score`. Then
look at the **mean review score in bands of `n_days_ahead`**. Does the correlation
describe this relationship well? What does the chart show that the single number does not?

In [ ]:
r = df['n_days_ahead'].corr(df['____'])
print(f"correlation {r:+.3f}\n")

bands = pd.cut(df['n_days_ahead'], bins=[-999, -15, -8, -1, 0, 7, 14, 21, 999],
               labels=['15+ late', '8-14 late', '1-7 late', 'on the day',
                       '1-7 early', '8-14 early', '15-21 early', '22+ early'])
by_band = df.groupby(bands, observed=True)['review_score'].agg(['count', 'mean']).round(2)
print(by_band)

by_band['mean'].plot(kind='bar', rot=30, ylabel='mean review score', xlabel='')
plt.show()

> 📊 **Slide 21** — the late vs on-time comparison, the correlation, the band chart, and your
> answer: does the correlation describe this relationship well?

---
## 7 · Late delivery and the handoff to the carrier

**Q7.1** Compare `n_days_to_carrier` for late and on-time orders: a boxplot, and the median and mean
of each group. Then compute the **late rate** in bands of `n_days_to_carrier`. Which shows the
difference better, the boxplot or the late rate by band?

In [ ]:
sns.boxplot(data=df, x='late', y='____')
plt.show()
print(df.groupby('late')['n_days_to_carrier'].agg(['median', 'mean']).round(1), "\n")

carrier_bands = pd.cut(df['n_days_to_carrier'], bins=[-999, 1, 2, 3, 5, 10, 999],
                       labels=['0-1', '2', '3', '4-5', '6-10', '11+'])
df.groupby(carrier_bands, observed=True)['late'].agg(n_orders='count', late_rate='____').round(3)

**Q7.2** Some orders have a **negative** `n_days_to_carrier`. How many? What would have to be true
for that to happen, and what would you do with these rows?

In [ ]:
print("negative n_days_to_carrier:", (df['n_days_to_carrier'] < ____).sum())
df[df['n_days_to_carrier'] < 0][['order_purchase_timestamp', 'order_delivered_carrier_date']].head()

> 📊 **Slide 22** — the comparison (chart and medians), the late rate by band, and the negative values.

---
## 8 · Late delivery and distance

**Q8.1** What is the late rate when the customer and the seller are in the same state, and when they
are not? The same city? Then: which customer states have the highest late rates? Use the 10 states
with the most orders, so each rate rests on enough orders.

In [ ]:
print(df.groupby('same_state')['late'].agg(n_orders='count', late_rate='mean').round(3), "\n")
print(df.groupby('____')['late'].agg(n_orders='count', late_rate='mean').round(3), "\n")

top_states = df['customer_state'].value_counts().index[:10]
state_rates = (df[df['customer_state'].isin(top_states)]
               .groupby('customer_state')['late'].mean().sort_values(ascending=False))
state_rates.plot(kind='barh', xlabel='late rate', ylabel='')
plt.show()

> 📊 **Slide 23** — the late rates by same state / same city, and the state chart.

---
## 9 · Late delivery, the promise, and the order itself

**Q9.1** How does the late rate change with the length of the promise? Compute the late rate for
each band of `n_days_promised` below (one week or less, 8–14 days, and so on), and plot it. Which
promises are most often late?

In [ ]:
promised_bands = pd.cut(df['n_days_promised'], bins=[0, 7, 14, 21, 28, 35, 45, 999],
                        labels=['1 week or less', '8-14 days', '15-21 days', '22-28 days',
                                '29-35 days', '36-45 days', '46+ days'])
by_promise = (df.groupby(promised_bands, observed=True)['late']
              .agg(n_orders='count', late_rate='____').round(3))
print(by_promise)

by_promise['late_rate'].plot(kind='bar', rot=30, ylabel='late rate', xlabel='days promised')
plt.show()

**Q9.2** Are bulky or heavy orders, or orders with a high shipping charge, more often late? Compute the
late rate by quintile of `total_weight_g`, `max_length_cm` and `freight`.

In [ ]:
for col in ['total_weight_g', 'max_length_cm', '____']:
    rates = df.groupby(pd.qcut(df[col], 5, duplicates='drop'), observed=True)['late'].mean()
    print(col, rates.round(3).tolist())

> 📊 **Slide 24** — the late rate by promise length, and by weight, size and freight.

---
## 10 · Late delivery and sellers

**Q10.1** Are late deliveries concentrated among a few sellers? Find the share of all late orders
that comes from the 10% of sellers with the most late orders, and compare it with the share of
**all** orders that the 10% largest sellers hold.

In [ ]:
late_by_seller = df[df['late'] == 1]['seller_id'].value_counts()
orders_by_seller = df['seller_id'].value_counts()

top_late = late_by_seller.head(int(len(orders_by_seller) * ____))
top_volume = orders_by_seller.head(int(len(orders_by_seller) * ____))
print(f"sellers: {len(orders_by_seller)}, of which {len(late_by_seller)} had a late order")
print(f"top 10% of sellers by late orders hold {top_late.sum() / late_by_seller.sum():.1%} of late orders")
print(f"top 10% of sellers by volume hold      {top_volume.sum() / orders_by_seller.sum():.1%} of all orders")

**Q10.2** For sellers with at least 10 orders, plot the distribution of their late rate. Is
lateness spread evenly, or do some sellers have far higher rates than the base rate?

In [ ]:
seller_rates = df.groupby('seller_id')['late'].agg(n_orders='count', late_rate='mean')
seller_rates = seller_rates[seller_rates['n_orders'] >= ____]
print(len(seller_rates), "sellers with 10+ orders")
print("For the sellers with 10+ orders, the late rate statistics are:")
print(seller_rates['late_rate'].describe().round(3))
sns.histplot(seller_rates['late_rate'], bins=30)
plt.axvline(df['late'].mean(), color='black', linestyle='--')
plt.xlabel('late rate of the seller (dashed line: base rate)')
plt.show()

**Q10.3** Are orders with more than one seller more often late? Compute the late rate by
`n_diff_sellers`. The result runs against intuition: give **two different explanations** for it, and
say whether this data can tell your two explanations apart.

In [ ]:
df.groupby('____')['late'].agg(n_orders='count', late_rate='mean').round(3)

> 📊 **Slide 25** — the two shares, the chart, your answer to whether lateness is concentrated, and
> the late rate by number of sellers with your two explanations.

---
## 11 · What to carry into the model

**Q11.1** Notebook 3 predicts `late` **at the moment the order is placed**. From Sections 6–10, list the
columns that look related to late delivery. Then sort them: which are known when the order is placed,
and which only exist later?

> 📊 **Slide 26** — two lists: promising columns known at purchase, and related columns that only
> exist after purchase.

---
### Before you submit

- [ ] Runtime → **Restart and run all** finishes with no errors
- [ ] Every 📊 slide above (14–26) is filled in the deck
- [ ] Your finding for slide 39, the team summary, is agreed with your team
- [ ] File → Download → **Download .ipynb**, and upload that file to Canvas. Download *after* the run above, so the file includes every output